# Detecção de alimentos com visão computacional

**Entrega 01: Inteligência Artificial e Aprendizado de Máquina (professor Victor)**

**Estudantes:**

1. André Gregório dos Santos – RA: 24026489
2. Guilherme Reis Fogolin de Godoy – RA: 24026241
3. Pedro Henrique Nascimento Lemos – RA: 23025380
4. Yan Ramos Cezareto – RA: 24026005


## 01. Classificação das imagens


O primeiro passo foi classificar as imagens através do Label Studio. Para isso, foram tiradas centenas de fotos em posições aleatórias para que, assim, fosse possível aumentar a confiabilidade da detecção.



<p align=center>
<img src="https://raw.githubusercontent.com/2026-1-NCC5/Projeto1/refs/heads/main/Imagens/Exemplo_Label_Studio.png" height="360"><br>
<i>Exemplo de classificação no Label Studio</i>
</p>

### **Alimentos**

Inicialmente, foram classificados três alimentos somente: **arroz**, **feijão** e **açúcar**, itens comuns de cesta básica.

Além disso, para promover maior confiabilidade nos testes porteriores foram usadas as seguintes marcas: **Namorado** (arroz), **Camil** (feijão) e **Melitta** (café).

### **Estrutura dos arquivos**

Com o uso do Label Studio para etiquetar e exportar as imagens, no final da classificação será possível gerar um arquivo `.zip` contendo o seguinte:

- Uma pasta `images` com as imagens;
- Uma pasta `labels` com os rótulos no formato de anotação YOLO;
- Um arquivo `classes.txt` com o mapa de rótulos contendo todas as classes;
- Um arquivo `notes.json` contendo informações específicas do Label Studio (este arquivo é ignorado para o nosso objetivo).

## 02. Importando dados e configurando o Google Colab


Vamos carregar nosso conjunto de dados e preparar o treinamento com o YOLO. Dividiremos o conjunto de dados em pastas de treinamento e validação e geraremos automaticamente o arquivo de configuração para treinar o modelo.

### 02.1 Configurando o Colab


O primeiro passo é ir nas configurações e alterar o tipo de ambiente de execução.

Para esse projetpo, será necessário selecionar o `GPUs: T4`.

<p align=center>
<img src="https://raw.githubusercontent.com/2026-1-NCC5/Projeto1/refs/heads/main/Imagens/Config_Colab.png" height="360"><br>
<i>Configurando o Google Colab</i>
</p>

Em seguida, rode o comando abaixo e verifique que aparece a descrição `Tesla T4`.

In [ ]:
# Validando disponibilidade de GPU

!nvidia-smi

### 02.2 Subindo arquivos de classificação

Como as classificações já foram efetuadas através do Label Studio, agora é preciso subir o arquivo `.zip`resultante na área de arquivos do Colab.

O arquivo está salvo em nosso repositório do Colab com o nome `dados_yolo.zip` e pode ser acessado através do link: [Projeto01](https://github.com/2026-1-NCC5/Projeto1/tree/main/Documentos/Entrega01/Intelig%C3%AAncia_Artifical).

<p align=center>
<img src="https://raw.githubusercontent.com/2026-1-NCC5/Projeto1/refs/heads/main/Imagens/Upload_Colab.png" height="360"><br>
<i>Fazendo upload de arquivo no Google Colab</i>
</p>

## 03. Dividindo imagens de treino e validação

Descompactando o arquivo `.zip`.

In [ ]:
!unzip -q /content/dados_yolo.zip -d /content/custom_data

Vamos usar no treino a biblioteca Ultralytics, a qual requer uma estrutura de pastas específica para armazenar os dados de treinamento dos modelos. A pasta raiz se chama `dados_yolo`. Dentro dela, existem duas pastas principais:


* `Train`: Estas são as imagens usadas para treinar o modelo. Em cada rodada de treinamento, todas as imagens do conjunto de treinamento são passadas para a rede neural. O algoritmo de treinamento ajusta os pesos da rede para que se adequem aos dados das imagens.

* `Validation`: Estas imagens são usadas para verificar o desempenho do modelo ao final de cada época de treinamento.

Dentro de cada uma dessas pastas, há uma pasta `images` e uma pasta `labels`, que contêm os arquivos de imagem e os arquivos de anotação, respectivamente.

O script Python abaixo vai criar automaticamente a estrutura acima citada, em que **90%** do dataset será para **treinamento** e **10%** apenas para **validação**.

In [ ]:
# Script de configuração

from pathlib import Path
import random
import os
import sys
import shutil

# Valores de treinamento

data_path = "/content/custom_data/dados_yolo"
train_percent = 0.9

# Validando entrada

if not os.path.isdir(data_path):
   print('O diretório especificado por --datapath não foi encontrado. Verifique se o caminho está correto e tente novamente.')
   sys.exit(0)
if train_percent < .01 or train_percent > 0.99:
   print('Entrada inválida para train_pct. Por favor, insira um número entre 0,01 e 0,99.')
   sys.exit(0)
val_percent = 1 - train_percent

# Define caminho de pastas de entrada

input_image_path = os.path.join(data_path,'images')
input_label_path = os.path.join(data_path,'labels')

# Definindo caminho para imagens e labels

cwd = os.getcwd()
train_img_path = os.path.join(cwd,'dados_yolo/train/images')
train_txt_path = os.path.join(cwd,'dados_yolo/train/labels')
val_img_path = os.path.join(cwd,'dados_yolo/validation/images')
val_txt_path = os.path.join(cwd,'dados_yolo/validation/labels')

# Criando pastas se ainda não existir

for dir_path in [train_img_path, train_txt_path, val_img_path, val_txt_path]:
   if not os.path.exists(dir_path):
      os.makedirs(dir_path)
      print(f'Pasta criada em: {dir_path}.')

# Lista de todas as imagens

img_file_list = [path for path in Path(input_image_path).rglob('*')]
txt_file_list = [path for path in Path(input_label_path).rglob('*')]

print(f'\nNúmero de imagens: {len(img_file_list)}')
print(f'Número de arquivos de anotação: {len(txt_file_list)}')

# Números de arquivos para cada pasta

file_num = len(img_file_list)
train_num = int(file_num*train_percent)
val_num = file_num - train_num
print('\nImagens movidas para treino: %d' % train_num)
print('Imagens movidas para validação: %d' % val_num)

# Seleção aleatória pasta teste e validação

for i, set_num in enumerate([train_num, val_num]):
  for ii in range(set_num):
    img_path = random.choice(img_file_list)
    img_fn = img_path.name
    base_fn = img_path.stem
    txt_fn = base_fn + '.txt'
    txt_path = os.path.join(input_label_path,txt_fn)

    if i == 0:
      new_img_path, new_txt_path = train_img_path, train_txt_path
    elif i == 1:
      new_img_path, new_txt_path = val_img_path, val_txt_path

    shutil.copy(img_path, os.path.join(new_img_path,img_fn))

    if os.path.exists(txt_path):
      shutil.copy(txt_path,os.path.join(new_txt_path,txt_fn))

    img_file_list.remove(img_path)

## 04. Instalando bibliotecas necessárias

Como já citado, a biblioteca Ultralytics será usada no treinamento.

In [ ]:
!pip install ultralytics

## 05 Configurando treinamento

Há um último passo antes de podermos executar o treinamento: precisamos criar o arquivo YAML de configuração de treinamento do Ultralytics. Este arquivo especifica a localização dos dados de treinamento e validação, e também define as classes do modelo.

O bloco de código abaixo vai gerar automaticamente um arquivo de configuração `dados_yolo.yaml`, em que:

1. Lê o arquivo `classes.txt` para obter a lista de nomes de classes (arros, feijão e açúcar);
2. Cria um dicionário de dados com os caminhos corretos para as pastas, o número de classes e os nomes das classes;
3. Escreve os dados em formato YAML no arquivo `dados_yolo.yaml`.

In [ ]:
import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  # Le o arquivo "classes.txt"

  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt não encontrado! Por favor, crie o classes.txt e mova para: {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Criando dicionário de dados

  data = {
      'path': '/content/dados_yolo',
      'train': 'train/images',
      'val': 'validation/images',
      'nc': number_of_classes,
      'names': classes
  }

  # Escreva o YAML file

  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Definindo caminhos

path_to_classes_txt = '/content/custom_data/dados_yolo/classes.txt'
path_to_data_yaml = '/content/dados_yolo.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nConteúdo do arquivo:\n')
!cat /content/dados_yolo.yaml

## 06. Rodando o treinamento

### 06.1 Definições

Com os dados organizados e o arquivo de configuração criado, o treinamento pode começar. Antes disso, é importante definir alguns parâmetros.

1. **Arquitetura e tamanho do modelo**

Existem vários tamanhos de modelos YOLO11 disponíveis: `yolo11n.pt`, `yolo11s.pt`, `yolo11m.pt`, `yolo11l.pt` e `yolo11xl.pt`. Modelos maiores têm maior precisão, mas rodam mais lentamente. Modelos menores rodam mais rápido, mas com menor precisão.

Para nosso objetivo, o modelo `yolo11s.pt` será empregado.

2. **Número de épocas**

Uma época é uma passagem completa por todo o conjunto de dados de treinamento. A quantidade ideal depende do tamanho dos dados: para conjuntos com menos de 200 imagens, 60 épocas é um bom ponto de partida. Para mais de 200 imagens, 40 épocas costuma funcionar bem.

Nesse trabalho, o teste com 70 época foi ideal, considerando o total de imagens usadas.


3. **Resolução**

A resolução impacta diretamente a velocidade e a precisão do modelo. O padrão para modelos YOLO é 640x640. Caso o objetivo seja um modelo mais rápido ou as imagens de entrada sejam de baixa resolução, vale testar com 480x480.

### 06.2. Rodando o treinamento

O algoritmo de treinamento analisará as imagens nos diretórios de treinamento e validação e, em seguida, iniciará o treinamento do modelo. Ao final de cada época de treinamento, o programa executa o modelo no conjunto de dados de validação e reporta o mAP, a precisão e o recall resultantes. Conforme o treinamento prossegue, o mAP geralmente aumenta a cada época. O treinamento será encerrado após atingir o número de épocas especificado por `epochs` (no caso, igual a 70).

> **Importante:**

**mAP significa Mean Average Precision** (Precisão Média). É a principal métrica usada para avaliar a qualidade de modelos de detecção de objetos como o YOLO.

1. **Como funciona?**

Ela combina duas métricas:

* Precisão — das detecções que o modelo fez, quantas estavam corretas?
* Recall — dos objetos reais nas imagens, quantos o modelo conseguiu encontrar?

O mAP calcula a média dessas métricas ao longo de diferentes limiares de confiança e de todas as classes detectadas, resultando em um único número.

2. **Como interpretar?**

| mAP | Interpretação |
| :--- | :--- |
| 0 a 0,4 | Fraco |
| 0,4 a 0,6 | Razoável |
| 0,6 a 0,8 | Bom |
| > 0,8 | Excelente |

O valor vai de 0 a 1 (ou 0% a 100%). Quanto mais próximo de 1, melhor o modelo está detectando os objetos corretamente.

3. **No contexto do treinamento**

Quando o log mostra o mAP aumentando a cada época, significa que o modelo está aprendendo progressivamente a identificar os alimentos com mais precisão. Se o mAP parar de subir ou começar a cair, pode ser sinal de que o modelo chegou no seu limite ou está sofrendo overfitting.

In [ ]:
# Rodar treinamento

!yolo detect train data=/content/dados_yolo.yaml model=yolo11s.pt epochs=70 imgsz=640

## 07. Validando os resultados

O modelo foi treinado e já pode ser testado! Os comandos abaixo executam o modelo nas imagens da pasta de validação e exibe os resultados das 25 primeiras imagens.

In [ ]:
!yolo detect predict model=runs/detect/train/weights/best.pt source=dados_yolo/validation/images save=True

In [ ]:
import glob
from IPython.display import Image, display
for image_path in glob.glob(f'/content/runs/detect/predict/*.jpg')[:25]:
  display(Image(filename=image_path, height=400))
  print('\n')

## 08. Testando modelo com uma câmera localmente

Como o Google Colab roda em servidores remotos do Google, ele não tem acesso direto à câmera do computador, o que impossibilita a captura de vídeo em tempo real. Por isso, após o treinamento, o modelo deve ser baixado e executado localmente, onde é possível acessar a câmera diretamente e realizar a detecção de alimentos em tempo real sem limitações de latência ou conectividade.

### 08.1 Download do modelo

In [ ]:
# Cria a pasta "modelo" para armazenar os pesos do modelo e os resultados do treinamento

!mkdir /content/modelo
!cp /content/runs/detect/train/weights/best.pt /content/modelo/modelo.pt
!cp -r /content/runs/detect/train /content/modelo

# Compactando o arquivo
%cd modelo
!zip /content/modelo.zip modelo.pt
!zip -r /content/modelo.zip train
%cd /content

# Faça o download através do menu lateral

### 08.2 Rodando localmente

Em seguida, vamos pegar o modelo baixado e executá-lo em um dispositivo local.

Com um script em Python, vamos carregar um modelo, executar inferência em uma imagem de origem, analisar os resultados da inferência e exibir caixas ao redor de cada classe detectada na imagem.

A forma mais fácil de rodar modelos Ultralytics em um PC é usando o Anaconda. O Anaconda configura um ambiente virtual Python e permite instalar o Ultralytics facilmente.

> **IMPORTANTE:** Execute os passos abaixo em seu computador local! Não use mais o Google Colab, pois não irá funcionar.

#### Passo 01


**Baixe e instale o Anaconda**

Acesse a página de download do Anaconda em https://anaconda.com/download, clique no botão "skip registration" e baixe o pacote para o seu sistema operacional. Após o download, execute o instalador e siga as etapas de instalação. As opções padrão podem ser mantidas.


#### Passo 02

**Configure o ambiente virtual**

Após a instalação, abra o Anaconda Prompt pelo Menu Iniciar.

Execute os seguintes comandos para criar um novo ambiente Python e ativá-lo:

```
conda create --name yolo-env1 python=3.11 -y
conda activate yolo-env1
```

Instale o Ultralytics (que também instala bibliotecas como OpenCV-Python, Numpy e PyTorch) com o seguinte comando:
```
pip install ultralytics
```

#### Passo 03


**Extraia o modelo baixado**

Pegue o arquivo `modelo.zip` baixado na etapa 08.1 e descompacte-o em uma pasta no seu PC. No terminal do Anaconda Prompt, navegue até a pasta descompactada com:

```
cd caminho/para/a/pasta
```

#### Passo 04

**Baixe e execute o yolo_detector.py**

Baixe o script `yolo_detector.py` para a pasta `modelo` com o comando:
```
curl -o yolo_detector.py https://raw.githubusercontent.com/2026-1-NCC5/Projeto1/refs/heads/main/Documentos/Entrega01/Intelig%C3%AAncia_Artifical/yolo_detector.py
```

Pronto! Agora é só rodar o script. Para executar a inferência com uma câmera USB, use:

```
python yolo_detect.py --model modelo.pt --source usb0 --resolution 720x480
```

> **Nota:** Em nosso caso, usamos uma câmera 720x480. Altere conforme o seu modelo.

Uma janela será aberta mostrando a transmissão ao vivo da sua webcam com caixas desenhadas ao redor dos objetos detectados em cada frame.

In [ ]:
!zip -r /content/resultados_treino.zip /content/runs/detect/train/

## 09. Conclusão

Conclusão do experimento na pasta do experimento no repositório do GitHub: [Projeto 01 - Inteligência Artificial e Aprendizado de Máquina](https://github.com/2026-1-NCC5/Projeto1/blob/main/Documentos/Entrega01/Intelig%C3%AAncia_Artifical/README.md).